# Sparse Continual Finetuning — 1p5b KL n256 v4

Runs **Stage 3** (`sparse_finetune.py`) on a Colab T4 GPU.

**What you need to upload (see Cell 3):**
- `background_stats.pt` (514 KB)
- `recovery/memory.pt` (51 MB)
- `data/recovery_eval_sample.jsonl` (tiny)
- `data/recovery_prompts.json` (tiny)

Training data streams from HuggingFace automatically.

## Cell 1 — Verify GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Cell 2 — Clone repo & install

In [ ]:
import os

# ---------------------------------------------------------------------------
# Option A: public repo or token-authenticated HTTPS
# Replace with your token if the repo is private:
#   GITHUB_TOKEN = 'ghp_xxxx'
#   repo_url = f'https://{GITHUB_TOKEN}@github.com/miguelaeh/continual-learning.git'
# ---------------------------------------------------------------------------
repo_url = 'https://github.com/miguelaeh/continual-learning.git'
repo_dir = '/content/continual-learning'

if not os.path.exists(repo_dir):
    !git clone {repo_url} {repo_dir}
else:
    print('Repo already cloned.')

# pyproject.toml is untracked locally so it wasn't pushed — write it here
pyproject = """\
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "smf-retrofit"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = [
  "datasets>=4.4.0",
  "PyYAML>=6.0",
  "torch>=2.5.0",
  "transformers>=4.57.0",
]

[tool.setuptools]
package-dir = {"" = "src"}

[tool.setuptools.packages.find]
where = ["src"]
"""
with open(f'{repo_dir}/6-april-paper/pyproject.toml', 'w') as f:
    f.write(pyproject)

%cd {repo_dir}/6-april-paper
!pip install -e . --quiet
print('Done.')

## Cell 3 — Upload checkpoints & data

Run this cell to upload files from your local machine into the right locations.
Upload these four files when prompted:
1. `background_stats.pt`  → goes to `experiments/1p5b_kl_n256_v4/checkpoints/`
2. `memory.pt` (recovery) → goes to `experiments/1p5b_kl_n256_v4/checkpoints/recovery/`
3. `recovery_eval_sample.jsonl` → goes to `data/`
4. `recovery_prompts.json` → goes to `data/`

In [ ]:
from google.colab import files
import shutil, os

repo_root = '/content/continual-learning/6-april-paper'
ckpt_dir  = f'{repo_root}/experiments/1p5b_kl_n256_v4/checkpoints'
rec_dir   = f'{ckpt_dir}/recovery'
data_dir  = f'{repo_root}/data'

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(rec_dir,  exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

print('Upload background_stats.pt')
up = files.upload()
for name, data in up.items():
    dest = f'{ckpt_dir}/background_stats.pt'
    open(dest, 'wb').write(data)
    print(f'  Saved → {dest}')

print('Upload memory.pt (recovery checkpoint)')
up = files.upload()
for name, data in up.items():
    dest = f'{rec_dir}/memory.pt'
    open(dest, 'wb').write(data)
    print(f'  Saved → {dest}')

print('Upload recovery_eval_sample.jsonl')
up = files.upload()
for name, data in up.items():
    dest = f'{data_dir}/recovery_eval_sample.jsonl'
    open(dest, 'wb').write(data)
    print(f'  Saved → {dest}')

print('Upload recovery_prompts.json')
up = files.upload()
for name, data in up.items():
    dest = f'{data_dir}/recovery_prompts.json'
    open(dest, 'wb').write(data)
    print(f'  Saved → {dest}')

print('All files in place.')

## Cell 3b — Alternative: copy from Google Drive

Skip this if you used Cell 3. Useful if you already have the files on Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# DRIVE_CKPT = '/content/drive/MyDrive/1p5b_kl_n256_v4/checkpoints'
# repo_root  = '/content/continual-learning/6-april-paper'
# ckpt_dir   = f'{repo_root}/experiments/1p5b_kl_n256_v4/checkpoints'
# rec_dir    = f'{ckpt_dir}/recovery'
# data_dir   = f'{repo_root}/data'
#
# import os, shutil
# os.makedirs(rec_dir, exist_ok=True)
# os.makedirs(data_dir, exist_ok=True)
# shutil.copy(f'{DRIVE_CKPT}/background_stats.pt',       f'{ckpt_dir}/background_stats.pt')
# shutil.copy(f'{DRIVE_CKPT}/recovery/memory.pt',        f'{rec_dir}/memory.pt')
# shutil.copy(f'{DRIVE_CKPT}/data/recovery_eval_sample.jsonl', f'{data_dir}/recovery_eval_sample.jsonl')
# shutil.copy(f'{DRIVE_CKPT}/data/recovery_prompts.json',      f'{data_dir}/recovery_prompts.json')
# print('Copied from Drive.')

## Cell 4 — Verify files are in place

In [ ]:
import os
repo_root = '/content/continual-learning/6-april-paper'
required = [
    'experiments/1p5b_kl_n256_v4/checkpoints/background_stats.pt',
    'experiments/1p5b_kl_n256_v4/checkpoints/recovery/memory.pt',
    'data/recovery_eval_sample.jsonl',
    'data/recovery_prompts.json',
    'experiments/1p5b_kl_n256_v4/continual.yaml',
]
all_ok = True
for f in required:
    path = os.path.join(repo_root, f)
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = f'OK ({size:,} bytes)' if exists else 'MISSING'
    print(f'  {status:>25}  {f}')
    if not exists:
        all_ok = False
print()
print('Ready to train!' if all_ok else 'Fix missing files before running training.')

## Cell 5 — Run sparse continual finetuning

In [ ]:
import subprocess, sys

repo_root = '/content/continual-learning/6-april-paper'
config    = 'experiments/1p5b_kl_n256_v4/continual.yaml'

cmd = [sys.executable, 'scripts/sparse_finetune.py', '--config', config]
print('Running:', ' '.join(cmd))
print('Working dir:', repo_root)
print()

proc = subprocess.Popen(
    cmd,
    cwd=repo_root,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'\nExit code: {proc.returncode}')

## Cell 6 — Download output checkpoint

In [ ]:
from google.colab import files
import os

repo_root  = '/content/continual-learning/6-april-paper'
output_dir = f'{repo_root}/experiments/1p5b_kl_n256_v4/checkpoints/continual'

ckpts = sorted([
    f for f in os.listdir(output_dir) if f.endswith('.pt')
])
print('Checkpoints available:', ckpts)

# Download final checkpoint
final = os.path.join(output_dir, 'memory.pt')
if os.path.exists(final):
    files.download(final)
else:
    print('memory.pt not found — training may not have finished. Download step checkpoints above.')
    for c in ckpts:
        files.download(os.path.join(output_dir, c))

## Cell 6b — Save to Google Drive instead

Alternative to Cell 6 if you prefer Drive storage.

In [ ]:
# from google.colab import drive
# import shutil, os
# drive.mount('/content/drive')
#
# src = '/content/continual-learning/6-april-paper/experiments/1p5b_kl_n256_v4/checkpoints/continual'
# dst = '/content/drive/MyDrive/1p5b_kl_n256_v4/continual'
# os.makedirs(dst, exist_ok=True)
# for f in os.listdir(src):
#     if f.endswith('.pt'):
#         shutil.copy(os.path.join(src, f), os.path.join(dst, f))
#         print(f'Copied {f} → Drive')